# Plant Disease Detection with Grad-CAM
EfficientNetB0, MobileNetV2, ResNet50 comparison on PlantVillage dataset.

In [ ]:
!pip install tensorflow opencv-python matplotlib pandas seaborn -q

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import cv2
import pandas as pd
print("TF version:", tf.__version__)

## Step 1: Download & Prepare Dataset

In [ ]:
from google.colab import files
files.upload()  # Upload kaggle.json

In [ ]:
!kaggle datasets download -d emmarex/plantdisease
!unzip -q plantdisease.zip
import os
print(os.listdir('/content/PlantVillage'))

In [ ]:
import os
dataset_path = "/content/PlantVillage"
for folder in os.listdir(dataset_path):
    folder_path = os.path.join(dataset_path, folder)
    if os.path.isdir(folder_path):
        print(folder, "->", len(os.listdir(folder_path)))

## Step 2: Load Dataset

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    "/content/PlantVillage",
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    "/content/PlantVillage",
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_ds.class_names
print("Classes:", class_names)
print("Total Classes:", len(class_names))

## Step 3: Data Augmentation

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomContrast(0.2)
])

## Step 4: Model 1 — EfficientNetB0 (Functional API for Grad-CAM support)

In [ ]:
# Using Functional API so Grad-CAM works correctly
eff_base = tf.keras.applications.EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)
eff_base.trainable = False

inputs = tf.keras.Input(shape=(224, 224, 3))
x = data_augmentation(inputs)
x = eff_base(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dense(256, activation='relu')(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(len(class_names), activation='softmax')(x)

model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5
)

In [ ]:
loss, eff_acc = model.evaluate(val_ds)
print("EfficientNetB0 Validation Accuracy:", round(eff_acc * 100, 2), "%")

In [ ]:
model.save('plant_disease_efficientnet.h5')

## Step 5: Model 2 — MobileNetV2

In [ ]:
mob_base = tf.keras.applications.MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)
mob_base.trainable = False

mobilenet_model = tf.keras.Sequential([
    data_augmentation,
    mob_base,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(len(class_names), activation='softmax')
])

mobilenet_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_mobile = mobilenet_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5
)

In [ ]:
loss, mobile_acc = mobilenet_model.evaluate(val_ds)
print("MobileNetV2 Accuracy:", round(mobile_acc * 100, 2), "%")

## Step 6: Model 3 — ResNet50

In [ ]:
res_base = tf.keras.applications.ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)
res_base.trainable = False

resnet_model = tf.keras.Sequential([
    data_augmentation,
    res_base,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(len(class_names), activation='softmax')
])

resnet_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_resnet = resnet_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5
)

In [ ]:
loss, resnet_acc = resnet_model.evaluate(val_ds)
print("ResNet50 Accuracy:", round(resnet_acc * 100, 2), "%")

## Step 7: Model Comparison

In [ ]:
results = {
    "EfficientNetB0": eff_acc,
    "MobileNetV2": mobile_acc,
    "ResNet50": resnet_acc
}

for name, acc in results.items():
    print(f"{name}: {round(acc * 100, 2)}%")

## Step 8: Predict on New Image

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
img_path = list(uploaded.keys())[0]

img = tf.keras.utils.load_img(img_path, target_size=(224, 224))
img_array = tf.keras.utils.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)   # shape: (1, 224, 224, 3)

prediction = model.predict(img_array)
predicted_class = np.argmax(prediction)
confidence = np.max(prediction)

print("Class:", class_names[predicted_class])
print("Confidence:", round(confidence * 100, 2), "%")

if confidence > 0.9:
    severity = "Severe"
elif confidence > 0.7:
    severity = "Moderate"
else:
    severity = "Mild"
print("Severity:", severity)

In [ ]:
# Top-3 predictions
pred = model.predict(img_array)[0]
top3 = np.argsort(pred)[-3:][::-1]
print("\nTop 3 Predictions:")
for idx in top3:
    print(f"  {class_names[idx]}: {round(pred[idx] * 100, 2)}%")

In [ ]:
recommendations = {
    "Tomato_Late_blight": {
        "treatment": "Apply Copper-based fungicide",
        "prevention": "Remove infected leaves and avoid overhead watering",
        "recovery": "7-14 days"
    },
    "Tomato_Early_blight": {
        "treatment": "Apply Chlorothalonil fungicide",
        "prevention": "Crop rotation and proper spacing",
        "recovery": "10-14 days"
    },
    "Potato___Late_blight": {
        "treatment": "Use Copper fungicide",
        "prevention": "Avoid excess moisture",
        "recovery": "7-14 days"
    },
    "Potato___Early_blight": {
        "treatment": "Apply Mancozeb fungicide",
        "prevention": "Avoid excess moisture",
        "recovery": "10-14 days"
    }
}

disease = class_names[predicted_class]
if disease in recommendations:
    print("\nTreatment:", recommendations[disease]["treatment"])
    print("Prevention:", recommendations[disease]["prevention"])
    print("Recovery Time:", recommendations[disease]["recovery"])
else:
    print("\nNo specific recommendation found for:", disease)

## Step 9: Grad-CAM Visualization

In [ ]:
def make_gradcam_heatmap(img_array, model, base_model, last_conv_layer_name):
    """
    Generate Grad-CAM heatmap.
    model: the full Functional model
    base_model: the EfficientNetB0 sub-model inside
    last_conv_layer_name: name of the conv layer to visualize
    """
    img_tensor = tf.convert_to_tensor(img_array, dtype=tf.float32)

    # Build grad model: inputs -> [conv layer output, final predictions]
    grad_model = tf.keras.models.Model(
        inputs=model.inputs,
        outputs=[
            base_model.get_layer(last_conv_layer_name).output,
            model.output
        ]
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_tensor, training=False)
        pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0)
    heatmap = heatmap / (tf.reduce_max(heatmap) + 1e-8)

    return heatmap.numpy()


def overlay_heatmap(img_path, heatmap, alpha=0.4):
    """Overlay Grad-CAM heatmap on original image."""
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    heatmap_resized = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
    heatmap_colored = np.uint8(255 * heatmap_resized)
    heatmap_colored = cv2.applyColorMap(heatmap_colored, cv2.COLORMAP_JET)
    heatmap_colored = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB)

    superimposed = cv2.addWeighted(img, 1 - alpha, heatmap_colored, alpha, 0)

    plt.figure(figsize=(8, 8))
    plt.imshow(superimposed)
    plt.title("Grad-CAM Heatmap")
    plt.axis("off")
    plt.show()

In [ ]:
# Run Grad-CAM on the uploaded image
last_conv_layer = "block7a_project_conv"

# img_array is already (1, 224, 224, 3) from prediction step above
# eff_base is the EfficientNetB0 sub-model
heatmap = make_gradcam_heatmap(img_array, model, eff_base, last_conv_layer)
overlay_heatmap(img_path, heatmap)